# Exercise 8: Multi-Tool -- Web Search + Code Interpreter

Combine `web_search` and `code_interpreter` in a single call so the model can:
1. Fetch live data from the web
2. Crunch numbers with Python in a sandboxed container

The model autonomously decides which tool to use at each step.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

## Send the multi-tool request

We ask the model to find real-time market cap data and then calculate percentage shares.

In [2]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[
        {"type": "web_search"},
        {"type": "code_interpreter", "container": {"type": "auto"}},
    ],
    instructions="Use web_search to find data, then use code_interpreter to analyze it. Show your work.",
    input=(
        "Find the current market caps of NVIDIA, Apple, and Microsoft. "
        "Then use code_interpreter to calculate each company's percentage share "
        "of the combined total, and determine how much NVIDIA would need to grow "
        "to match Apple's market cap."
    ),
)

## Inspect the tool-use chain

See the sequence of web searches, code execution, and final message.

In [3]:
print("=== Tool usage chain ===\n")
for i, item in enumerate(response.output):
    if item.type == "web_search_call":
        print(f"[{i}] WEB_SEARCH (id={item.id[:20]}...)")
    elif item.type == "code_interpreter_call":
        code_preview = item.code.strip().split("\n")[0]
        print(f"[{i}] CODE_INTERPRETER: {code_preview}...")
    elif item.type == "message":
        text = ""
        for c in item.content:
            if c.type == "output_text":
                text = c.text[:80]
        print(f"[{i}] MESSAGE: {text}...")

=== Tool usage chain ===

[0] WEB_SEARCH (id=ws_08ed08fe30ab573e0...)
[1] MESSAGE: Here are the current market capitalizations as of today, Tuesday, March 3, 2026:...
... CODE_INTERPRETER: # Market caps in trillions of USD
[3] MESSAGE: Here are the calculations based on the latest market caps:

- Combined market ca...


## Full response

In [4]:
print("=== Full response ===\n")
for item in response.output:
    if item.type == "message":
        for c in item.content:
            if c.type == "output_text":
                print(c.text)
                print()

=== Full response ===

Here are the current market capitalizations as of today, Tuesday, March 3, 2026:

- NVIDIA (NVDA): approximately \$4.526 trillion citeturn0finance0  
- Apple (AAPL): approximately \$4.049 trillion citeturn0finance1  
- Microsoft (MSFT): approximately \$3.594 trillion citeturn0finance2  

---

## Combined Market Cap & Individual Percentage Shares

Let’s calculate each company's share percentage of the combined total and determine how much NVIDIA would need to grow to match Apple’s market cap.

Using Python in the code interpreter for precision:

```python
# Market caps in trillions of USD
nvda = 4.526118  # NVIDIA
aapl = 4.04915133  # Apple
msft = 3.59444648  # Microsoft

combined = nvda + aapl + msft

nvda_share = (nvda / combined) * 100
aapl_share = (aapl / combined) * 100
msft_share = (msft / combined) * 100

growth_needed = ((aapl - nvda) / nvda) * 100 if nvda < aapl else 0

nvda, aapl, msft, combined, nvda_share, aapl_share, msft_share, growth_needed

In [5]:
print(f"--- Tokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out ---")
print(f"--- Output items: {len(response.output)} ---")

--- Tokens: 1395 in, 901 out ---
--- Output items: 4 ---


---

**Interview one-liner:** Passing multiple built-in tools (`web_search`, `code_interpreter`) in one request lets the model orchestrate a research-then-compute workflow without any custom glue code.